In [ ]:
import pandas as pd

df = pd.read_csv("/content/cleaned_russia_ukraine_dataset.xls")
df.shape

(1458106, 7)

In [ ]:
df.isnull().sum()

,0
lang,0
country,0
reconstructed_text,12
main_sentiment,0
sentiment_score,0
main_stance,0
stance_score,0


In [ ]:
null_indices = df[df.isnull().any(axis=1)].index.tolist()

print(len(null_indices))
print(null_indices)

12
[511963, 519333, 719242, 1157238, 1157239, 1270055, 1345190, 1345191, 1345192, 1345193, 1345194, 1345195]


In [ ]:
df = df.dropna(subset=["reconstructed_text"])

print(df.shape)

(1458094, 7)


In [ ]:
df["main_stance"].value_counts(normalize=True) * 100

,proportion
main_stance,
Unsure,90.530926
Pro Ukraine,7.787838
Pro Russia,1.681236


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/NLP_Project_Preprocessing"

os.makedirs(SAVE_DIR, exist_ok=True)

Mounted at /content/drive


In [ ]:
pip install joblib

In [ ]:
from sklearn.preprocessing import LabelEncoder
import joblib

stance_encoder = LabelEncoder()

df["stance_encoded"] = stance_encoder.fit_transform(
    df["main_stance"]
)

print("Stance Mapping:")
print(dict(zip(
    stance_encoder.classes_,
    stance_encoder.transform(stance_encoder.classes_)
)))

joblib.dump(
    stance_encoder,
    f"{SAVE_DIR}/stance_encoder.pkl"
)

Stance Mapping:
{'Pro Russia': np.int64(0), 'Pro Ukraine': np.int64(1), 'Unsure': np.int64(2)}


['/content/drive/MyDrive/NLP_Project_Preprocessing/stance_encoder.pkl']

In [ ]:
sentiment_encoder = LabelEncoder()

df["sentiment_encoded"] = sentiment_encoder.fit_transform(
    df["main_sentiment"]
)

print("\nSentiment Mapping:")
print(dict(zip(
    sentiment_encoder.classes_,
    sentiment_encoder.transform(
        sentiment_encoder.classes_
    )
)))

joblib.dump(
    sentiment_encoder,
    f"{SAVE_DIR}/sentiment_encoder.pkl"
)


Sentiment Mapping:
{'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}


['/content/drive/MyDrive/NLP_Project_Preprocessing/sentiment_encoder.pkl']

In [ ]:
df["stratify_label"] = (
    df["main_stance"].astype(str)
    + "_"
    + df["main_sentiment"].astype(str)
)

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

X = df["reconstructed_text"]

y_stance = df["stance_encoded"]
y_sentiment = df["sentiment_encoded"]

indices = np.arange(len(df))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.2,
    random_state=42,
    stratify=df["stratify_label"]
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train_stance = y_stance.iloc[train_idx]
y_test_stance = y_stance.iloc[test_idx]

y_train_sentiment = y_sentiment.iloc[train_idx]
y_test_sentiment = y_sentiment.iloc[test_idx]

In [ ]:
import numpy as np

np.save(f"{SAVE_DIR}/train_idx.npy", train_idx)
np.save(f"{SAVE_DIR}/test_idx.npy", test_idx)

np.save(f"{SAVE_DIR}/y_train_stance.npy", y_train_stance)
np.save(f"{SAVE_DIR}/y_test_stance.npy", y_test_stance)

np.save(f"{SAVE_DIR}/y_train_sentiment.npy", y_train_sentiment)
np.save(f"{SAVE_DIR}/y_test_sentiment.npy", y_test_sentiment)

In [ ]:
X_train.to_csv(
    f"{SAVE_DIR}/X_train.csv",
    index=False
)

X_test.to_csv(
    f"{SAVE_DIR}/X_test.csv",
    index=False
)

In [ ]:
import nltk
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
from nltk.tokenize import word_tokenize

X_train_tokens = X_train.apply(word_tokenize)
X_test_tokens = X_test.apply(word_tokenize)

In [ ]:
import pickle

with open(f"{SAVE_DIR}/X_train_tokens.pkl", "wb") as f:
    pickle.dump(X_train_tokens, f)

with open(f"{SAVE_DIR}/X_test_tokens.pkl", "wb") as f:
    pickle.dump(X_test_tokens, f)

In [ ]:
print(X_train.iloc[0])

print(word_tokenize(X_train.iloc[0]))

ukraine rejects russian neutrality proposals
['ukraine', 'rejects', 'russian', 'neutrality', 'proposals']


In [ ]:
pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 24.8 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    sg=1
)

In [ ]:
w2v_model.save(f"{SAVE_DIR}/word2vec.model")

In [ ]:
print(w2v_model.wv["russia"][:10])

[-0.12483346  0.18569712  0.39113492 -0.12407362  0.34516174 -0.22990179
  0.02869383  0.23310585 -0.20271012 -0.29058763]


In [ ]:
len(w2v_model.wv.key_to_index)

81956

In [ ]:
word_index = {
    word: idx + 2
    for idx, word in enumerate(w2v_model.wv.key_to_index)
}

vocab_size = len(word_index) + 2

In [ ]:
import pickle

with open(f"{SAVE_DIR}/word_index.pkl", "wb") as f:
    pickle.dump(word_index, f)

In [ ]:
vocab_size

81958

In [ ]:
def tokens_to_sequence(tokens):
    return [word_index.get(word, 1) for word in tokens]

X_train_seq = X_train_tokens.apply(tokens_to_sequence)
X_test_seq = X_test_tokens.apply(tokens_to_sequence)

In [ ]:
print(X_train_tokens.iloc[0])
print(X_train_seq.iloc[0])

['ukraine', 'rejects', 'russian', 'neutrality', 'proposals']
[2, 1292, 4, 1701, 2651]


In [ ]:
lengths = X_train_seq.apply(len)

print("Max:", lengths.max())
print("Mean:", lengths.mean())
print("95th percentile:", lengths.quantile(0.95))
print("99th percentile:", lengths.quantile(0.99))

Max: 88
Mean: 13.141543539295741
95th percentile: 25.0
99th percentile: 30.0


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_LEN = 30

X_train_pad = pad_sequences(
    X_train_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

X_test_pad = pad_sequences(
    X_test_seq,
    maxlen=MAX_LEN,
    padding="post",
    truncating="post"
)

In [ ]:
print(X_train_pad.shape)
print(X_test_pad.shape)

(1166475, 30)
(291619, 30)


In [ ]:
np.save(f"{SAVE_DIR}/X_train_pad.npy", X_train_pad)
np.save(f"{SAVE_DIR}/X_test_pad.npy", X_test_pad)

In [ ]:
import numpy as np

embedding_dim = w2v_model.vector_size

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in word_index.items():
    embedding_matrix[idx] = w2v_model.wv[word]

In [ ]:
print(embedding_matrix.shape)

(81958, 100)


In [ ]:
np.save(
    f"{SAVE_DIR}/embedding_matrix.npy",
    embedding_matrix
)

In [ ]:
print(os.listdir(SAVE_DIR))

['word2vec.model', 'embedding_matrix.npy', 'X_train_pad.npy', 'X_test_pad.npy', 'word_index.pkl', 'stance_encoder.pkl', 'sentiment_encoder.pkl', 'train_idx.npy', 'test_idx.npy', 'y_train_stance.npy', 'y_test_stance.npy', 'y_train_sentiment.npy', 'y_test_sentiment.npy', 'X_train.csv', 'X_test.csv', 'X_train_tokens.pkl', 'X_test_tokens.pkl']
